# start

In [ ]:
# ============================================================
# Experiment E revised: Kernel Ablation for IntervalGP-VAE
# Rewritten using the same IntervalGP-VAE framework and parameters
# as the second uploaded IntervalGP-VAE implementation.
#
# Compared kernels kept from the first code:
#   1) RBF
#   2) Matern 1/2
#   3) Matern 3/2
#   4) Matern 5/2
#   5) Rational Quadratic
#   6) Linear + RBF
#
# Main framework matched to the second code:
#   - chosen_version = "u_aux"
#   - latent_dim = 1
#   - hidden_dim = 64
#   - causal_head_hidden_dim = 64
#   - GP_LENGTHSCALE = 7
#   - GP_VARIANCE = 2.0
#   - GP_NOISE = 1e-4
#   - BATCH_SIZE = 128
#   - JOINT_EPOCHS = 200
#   - HEAD_EPOCHS = 100
#   - VAE_REFINE_EPOCHS = 50
#   - AdamW optimizer with weight_decay = 1e-5
#   - three-stage training
#   - latent GP posterior smoothing
#   - ITE-space IntervalGP full-GP posterior inference

In [ ]:
# ============================================================

import os
import time
import math
import random
import warnings
import itertools
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader

warnings.filterwarnings("ignore")

In [ ]:
# ============================================================
# 0. Basic settings: matched to the second IntervalGP-VAE code

In [ ]:
# ============================================================

torch.set_default_tensor_type("torch.FloatTensor")

OUTPUT_DIR = "experiment_E_kernel_ablation_second_framework_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

DEVICE = "cpu"
GLOBAL_SEED = 420

np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)
random.seed(GLOBAL_SEED)

# Set to a small number, e.g. 2, for debugging.
# Set to None to run all selected synthetic settings.
MAX_ITERATIONS = None

chosen_version = "u_aux"

# Main IntervalGP-VAE hyperparameters from the second code
INTERVALGPVAE_LATENT_DIM = 1
INTERVALGPVAE_HIDDEN_DIM = 64
CAUSAL_HEAD_HIDDEN_DIM = 64

GP_LENGTHSCALE = 7
GP_VARIANCE = 2.0
GP_NOISE = 1e-4

BATCH_SIZE = 128
JOINT_EPOCHS = 200
HEAD_EPOCHS = 100
VAE_REFINE_EPOCHS = 50

LR_JOINT = 1e-3
LR_HEAD = 1e-3
LR_VAE_REFINE = 1e-4
WEIGHT_DECAY = 1e-5

ITE_CI = 0.90
CI = ITE_CI
Z_VALUE_90 = 1.6448536269514722
N_ITE_SAMPLES = 100

# If GP is too slow, set MAX_GP_POINTS = 300 or 500.
# If you want full GP over all training samples, set MAX_GP_POINTS = None.
MAX_GP_POINTS = None

# Synthetic experiment size, kept from the first code.
N_TRAIN = 1000
N_TEST = 200
NOISE_STD = 0.1

RUN_SYNTHETIC = True

In [ ]:
# ============================================================
# 1. Utility functions

In [ ]:
# ============================================================

def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    random.seed(seed)


def to_numpy_1d(x):
    if torch.is_tensor(x):
        x = x.detach().cpu().numpy()
    return np.asarray(x).reshape(-1)


def safe_corr(x, y):
    x = to_numpy_1d(x)
    y = to_numpy_1d(y)

    valid = np.isfinite(x) & np.isfinite(y)

    if valid.sum() < 2:
        return np.nan

    if np.std(x[valid]) == 0 or np.std(y[valid]) == 0:
        return np.nan

    return float(np.corrcoef(x[valid], y[valid])[0, 1])


def pehe_np(est_ite, true_ite):
    est_ite = to_numpy_1d(est_ite)
    true_ite = to_numpy_1d(true_ite)

    valid = np.isfinite(est_ite) & np.isfinite(true_ite)

    if valid.sum() == 0:
        return np.nan

    return float(np.sqrt(np.mean((est_ite[valid] - true_ite[valid]) ** 2)))


def compute_interval_metrics(lower, upper, true_value):
    lower = to_numpy_1d(lower)
    upper = to_numpy_1d(upper)
    true_value = to_numpy_1d(true_value)

    valid = (
        np.isfinite(lower)
        & np.isfinite(upper)
        & np.isfinite(true_value)
    )

    if valid.sum() == 0:
        return {
            "coverage": np.nan,
            "avg_length": np.nan,
            "n": 0,
        }

    covered = (
        (lower[valid] <= true_value[valid])
        & (true_value[valid] <= upper[valid])
    )

    return {
        "coverage": float(np.mean(covered)),
        "avg_length": float(np.mean(upper[valid] - lower[valid])),
        "n": int(valid.sum()),
    }


def interval_score_np(lower, upper, true_value, alpha=0.10):
    lower = to_numpy_1d(lower)
    upper = to_numpy_1d(upper)
    true_value = to_numpy_1d(true_value)

    valid = (
        np.isfinite(lower)
        & np.isfinite(upper)
        & np.isfinite(true_value)
    )

    if valid.sum() == 0:
        return np.nan

    l = lower[valid]
    u = upper[valid]
    y = true_value[valid]

    width = u - l
    below = y < l
    above = y > u

    score = (
        width
        + (2.0 / alpha) * (l - y) * below
        + (2.0 / alpha) * (y - u) * above
    )

    return float(np.mean(score))


def approximate_nll_np(point, lower, upper, true_value):
    point = to_numpy_1d(point)
    lower = to_numpy_1d(lower)
    upper = to_numpy_1d(upper)
    true_value = to_numpy_1d(true_value)

    valid = (
        np.isfinite(point)
        & np.isfinite(lower)
        & np.isfinite(upper)
        & np.isfinite(true_value)
    )

    if valid.sum() == 0:
        return np.nan

    point = point[valid]
    lower = lower[valid]
    upper = upper[valid]
    true_value = true_value[valid]

    std = (upper - lower) / (2.0 * Z_VALUE_90)
    std = np.maximum(std, 1e-6)

    nll = 0.5 * np.log(2.0 * np.pi * std ** 2) + 0.5 * ((true_value - point) / std) ** 2

    return float(np.mean(nll))


def align_recovered_u_to_true_u(u_recovered, u_true):
    u_recovered = to_numpy_1d(u_recovered)
    u_true = to_numpy_1d(u_true)

    valid = np.isfinite(u_recovered) & np.isfinite(u_true)

    if valid.sum() < 2:
        return u_recovered, np.nan, np.nan, np.nan, np.nan

    x = u_recovered[valid]
    y = u_true[valid]

    A = np.vstack([x, np.ones_like(x)]).T
    a, b = np.linalg.lstsq(A, y, rcond=None)[0]

    u_aligned = a * u_recovered + b
    corr = safe_corr(u_aligned, u_true)
    rmse = float(np.sqrt(np.mean((u_aligned[valid] - u_true[valid]) ** 2)))

    return u_aligned, float(a), float(b), corr, rmse


def maybe_subsample_gp_reference(x, *ys, max_points=None, seed=0):
    n = x.shape[0]

    if max_points is None or n <= max_points:
        return (x, *ys)

    rng = np.random.default_rng(seed)
    idx = rng.choice(np.arange(n), size=max_points, replace=False)
    idx = np.sort(idx)
    idx_t = torch.tensor(idx, dtype=torch.long, device=x.device)

    out = [x[idx_t]]

    for y in ys:
        if torch.is_tensor(y):
            out.append(y[idx_t])
        else:
            out.append(np.asarray(y)[idx])

    return tuple(out)

In [ ]:
# ============================================================
# 2. Kernel functions: compared methods from the first code

In [ ]:
# ============================================================

def _prepare_kernel_input(x):
    if x.ndim == 1:
        x = x.view(-1, 1)
    return x.float()


def pairwise_sqdist(x1, x2):
    x1 = _prepare_kernel_input(x1)
    x2 = _prepare_kernel_input(x2)
    return torch.cdist(x1, x2).pow(2)


def pairwise_dist(x1, x2):
    return torch.sqrt(pairwise_sqdist(x1, x2) + 1e-12)


def kernel_matrix(
    x1,
    x2=None,
    kernel_name="RBF",
    lengthscale=GP_LENGTHSCALE,
    variance=GP_VARIANCE,
    alpha=1.0,
    linear_offset=1.0,
):
    x1 = _prepare_kernel_input(x1)

    if x2 is None:
        x2 = x1
    else:
        x2 = _prepare_kernel_input(x2)

    lengthscale_t = torch.as_tensor(lengthscale, dtype=x1.dtype, device=x1.device)
    variance_t = torch.as_tensor(variance, dtype=x1.dtype, device=x1.device)

    if kernel_name == "RBF":
        d2 = pairwise_sqdist(x1, x2)
        K = variance_t * torch.exp(-0.5 * d2 / (lengthscale_t ** 2))

    elif kernel_name == "Matern12":
        r = pairwise_dist(x1, x2)
        K = variance_t * torch.exp(-r / lengthscale_t)

    elif kernel_name == "Matern32":
        r = pairwise_dist(x1, x2)
        sqrt3 = math.sqrt(3.0)
        scaled = sqrt3 * r / lengthscale_t
        K = variance_t * (1.0 + scaled) * torch.exp(-scaled)

    elif kernel_name == "Matern52":
        r = pairwise_dist(x1, x2)
        sqrt5 = math.sqrt(5.0)
        scaled = sqrt5 * r / lengthscale_t
        K = variance_t * (1.0 + scaled + scaled.pow(2) / 3.0) * torch.exp(-scaled)

    elif kernel_name == "RationalQuadratic":
        d2 = pairwise_sqdist(x1, x2)
        alpha_t = torch.as_tensor(alpha, dtype=x1.dtype, device=x1.device)
        K = variance_t * torch.pow(
            1.0 + d2 / (2.0 * alpha_t * lengthscale_t ** 2),
            -alpha_t,
        )

    elif kernel_name == "LinearPlusRBF":
        d2 = pairwise_sqdist(x1, x2)
        K_rbf = variance_t * torch.exp(-0.5 * d2 / (lengthscale_t ** 2))

        # Use a global centering approximation for cross-covariance.
        x1_centered = x1 - x1.mean(dim=0, keepdim=True)
        x2_centered = x2 - x2.mean(dim=0, keepdim=True)
        K_linear = (x1_centered @ x2_centered.t()) / max(1, x1.shape[1])
        K = K_rbf + linear_offset * K_linear

    else:
        raise ValueError(f"Unknown kernel_name: {kernel_name}")

    return K


KERNEL_CONFIGS = [
    {"kernel_name": "RBF", "display_name": "RBF", "alpha": 1.0, "linear_offset": 1.0},
    {"kernel_name": "Matern12", "display_name": "Matérn 1/2", "alpha": 1.0, "linear_offset": 1.0},
    {"kernel_name": "Matern32", "display_name": "Matérn 3/2", "alpha": 1.0, "linear_offset": 1.0},
    {"kernel_name": "Matern52", "display_name": "Matérn 5/2", "alpha": 1.0, "linear_offset": 1.0},
    {"kernel_name": "RationalQuadratic", "display_name": "Rational Quadratic", "alpha": 1.0, "linear_offset": 1.0},
    {"kernel_name": "LinearPlusRBF", "display_name": "Linear + RBF", "alpha": 1.0, "linear_offset": 1.0},
]

In [ ]:
# ============================================================
# 3. Synthetic data-generating settings from the first code

In [ ]:
# ============================================================

proxy_func_sets = [
    [
        lambda u, ua0, ua1: u,
        lambda u, ua0, ua1: u ** 2,
        lambda u, ua0, ua1: u ** 3,
        lambda u, ua0, ua1: torch.sin(u),
        lambda u, ua0, ua1: torch.tanh(u),
        lambda u, ua0, ua1: torch.exp(-0.5 * u ** 2),
    ],
    [
        lambda u, ua0, ua1: u,
        lambda u, ua0, ua1: torch.sin(u),
        lambda u, ua0, ua1: torch.tanh(0.5 * u),
        lambda u, ua0, ua1: torch.exp(0.3 * u),
        lambda u, ua0, ua1: torch.log1p(u ** 2),
        lambda u, ua0, ua1: torch.sigmoid(u),
    ],
    [
        lambda u, ua0, ua1: u,
        lambda u, ua0, ua1: u ** 2,
        lambda u, ua0, ua1: torch.sin(u),
        lambda u, ua0, ua1: torch.tanh(u),
        lambda u, ua0, ua1: torch.exp(-0.2 * u ** 2),
        lambda u, ua0, ua1: torch.sin(2 * u),
    ],
    [
        lambda u, ua0, ua1: u,
        lambda u, ua0, ua1: u ** 3,
        lambda u, ua0, ua1: torch.tanh(u),
        lambda u, ua0, ua1: torch.sigmoid(u),
        lambda u, ua0, ua1: torch.log1p(u ** 2),
        lambda u, ua0, ua1: torch.sin(u),
    ],
]


def treat_func_linear_1(u_np, ua0_np=None):
    logits = 0.8 * u_np
    probs = 1.0 / (1.0 + np.exp(-logits))
    return np.random.binomial(1, probs).astype(np.float32).reshape(-1)


def treat_func_linear_2(u_np, ua0_np=None):
    logits = 0.4 * u_np + 0.3
    probs = 1.0 / (1.0 + np.exp(-logits))
    return np.random.binomial(1, probs).astype(np.float32).reshape(-1)


def outcome_func_1(u_np, t_np, ua0=None, ua1=None, noise_std=0.1):
    if t_np.ndim > 1:
        t_np = t_np.reshape(-1)

    base = (
        0.5 * u_np
        + 0.2 * np.sin(u_np)
        + t_np.reshape(-1, 1) * (0.8 + 0.3 * u_np)
    )

    noise = np.random.normal(0.0, noise_std, size=base.shape)
    return (base + noise).astype(np.float32)


def outcome_func_2(u_np, t_np, ua0=None, ua1=None, noise_std=0.1):
    if t_np.ndim > 1:
        t_np = t_np.reshape(-1)

    base = (
        0.5 * u_np
        + 0.3 * np.cos(u_np)
        + t_np.reshape(-1, 1) * (0.5 + 0.5 * u_np)
    )

    noise = np.random.normal(0.0, noise_std, size=base.shape)
    return (base + noise).astype(np.float32)


SYNTHETIC_SETTINGS = [
    {
        "setting_id": "S1",
        "proxy_funcs": proxy_func_sets[0],
        "treatment_func": treat_func_linear_1,
        "outcome_func": outcome_func_1,
    },
    {
        "setting_id": "S2",
        "proxy_funcs": proxy_func_sets[1],
        "treatment_func": treat_func_linear_1,
        "outcome_func": outcome_func_2,
    },
    {
        "setting_id": "S3",
        "proxy_funcs": proxy_func_sets[2],
        "treatment_func": treat_func_linear_2,
        "outcome_func": outcome_func_1,
    },
    {
        "setting_id": "S4",
        "proxy_funcs": proxy_func_sets[3],
        "treatment_func": treat_func_linear_2,
        "outcome_func": outcome_func_2,
    },
]


def generate_synthetic_data(
    n=1000,
    noise_std=0.1,
    seed=0,
    proxy_funcs=None,
    treatment_func=None,
    outcome_func=None,
):
    set_seed(seed)

    u_np = np.random.normal(0, 1, size=(n, 1)).astype(np.float32)
    ua0_np = np.random.normal(0, 1, size=(n, 1)).astype(np.float32)
    ua1_np = np.random.normal(0, 1, size=(n, 1)).astype(np.float32)

    u = torch.from_numpy(u_np)
    ua0 = torch.from_numpy(ua0_np)
    ua1 = torch.from_numpy(ua1_np)

    if proxy_funcs is None:
        proxy_funcs = proxy_func_sets[0]

    clean_z = torch.cat([g(u, ua0, ua1) for g in proxy_funcs], dim=1)
    z = clean_z + torch.randn_like(clean_z) * noise_std

    if treatment_func is None:
        treatment_func = treat_func_linear_1

    t_np = treatment_func(u_np, ua0_np)
    t = torch.from_numpy(t_np).float()

    if outcome_func is None:
        outcome_func = outcome_func_1

    y_np = outcome_func(u_np, t_np, ua0_np, ua1_np, noise_std=noise_std)
    y = torch.from_numpy(y_np.squeeze()).float()

    return {
        "z": z.float(),
        "t": t.float(),
        "y": y.float(),
        "u": u.squeeze().float(),
        "ua0": ua0.squeeze().float(),
        "ua1": ua1.squeeze().float(),
        "proxy_funcs": proxy_funcs,
        "treatment_func": treatment_func,
        "outcome_func": outcome_func,
    }


def true_ite_from_outcome(outcome_func, u_tensor):
    u_np = u_tensor.detach().cpu().numpy().reshape(-1, 1)
    t0_np = np.zeros((len(u_np),), dtype=np.float32)
    t1_np = np.ones((len(u_np),), dtype=np.float32)

    if outcome_func.__name__ == "outcome_func_1":
        tau = 0.8 + 0.3 * u_np
    elif outcome_func.__name__ == "outcome_func_2":
        tau = 0.5 + 0.5 * u_np
    else:
        y0 = outcome_func(u_np, t0_np, None, None, noise_std=0.0)
        y1 = outcome_func(u_np, t1_np, None, None, noise_std=0.0)
        tau = y1 - y0

    return torch.from_numpy(tau.squeeze()).float()

In [ ]:
# ============================================================
# 4. Model components from the second IntervalGP-VAE framework

In [ ]:
# ============================================================

class DoubleEncoder(nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_dim,
        latent_dim,
        use_auxiliary_latents=False,
    ):
        super().__init__()

        self.use_aux = use_auxiliary_latents

        self.fc1 = nn.Linear(input_dim, hidden_dim)

        self.mean_u = nn.Linear(hidden_dim, latent_dim)
        self.logvar_u = nn.Linear(hidden_dim, latent_dim)

        self.mean_eps = nn.Linear(hidden_dim, input_dim)
        self.logvar_eps = nn.Linear(hidden_dim, input_dim)

        if self.use_aux:
            self.mean_ua0 = nn.Linear(hidden_dim, latent_dim)
            self.logvar_ua0 = nn.Linear(hidden_dim, latent_dim)

            self.mean_ua1 = nn.Linear(hidden_dim, latent_dim)
            self.logvar_ua1 = nn.Linear(hidden_dim, latent_dim)

    def forward(self, z):
        h = F.relu(self.fc1(z))

        mu_u = self.mean_u(h)
        logvar_u = self.logvar_u(h)
        std_u = torch.exp(0.5 * logvar_u)

        mu_eps = self.mean_eps(h)
        logvar_eps = self.logvar_eps(h)
        std_eps = torch.exp(0.5 * logvar_eps)

        if self.use_aux:
            mu_ua0 = self.mean_ua0(h)
            logvar_ua0 = self.logvar_ua0(h)
            std_ua0 = torch.exp(0.5 * logvar_ua0)

            mu_ua1 = self.mean_ua1(h)
            logvar_ua1 = self.logvar_ua1(h)
            std_ua1 = torch.exp(0.5 * logvar_ua1)

            return (
                mu_u,
                std_u,
                logvar_u,
                mu_eps,
                std_eps,
                logvar_eps,
                mu_ua0,
                std_ua0,
                logvar_ua0,
                mu_ua1,
                std_ua1,
                logvar_ua1,
            )

        return mu_u, std_u, logvar_u, mu_eps, std_eps, logvar_eps


class AdditiveDecoder(nn.Module):
    def __init__(
        self,
        latent_dim,
        hidden_dim,
        output_dim,
        use_auxiliary_latents=False,
    ):
        super().__init__()

        self.use_aux = use_auxiliary_latents
        factor = 1 + (2 if use_auxiliary_latents else 0)

        self.fc1 = nn.Linear(latent_dim * factor, hidden_dim)
        self.out = nn.Linear(hidden_dim, output_dim)

    def forward(self, u, eps, ua0=None, ua1=None):
        if self.use_aux:
            x = torch.cat([u, ua0, ua1], dim=1)
        else:
            x = u

        h = F.relu(self.fc1(x))
        return self.out(h) + eps


class GPVAEwithNoise(nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_dim,
        latent_dim,
        kernel_name="RBF",
        gp_lengthscale=GP_LENGTHSCALE,
        gp_variance=GP_VARIANCE,
        gp_noise=GP_NOISE,
        use_auxiliary_latents=False,
        rq_alpha=1.0,
        linear_offset=1.0,
    ):
        super().__init__()

        self.use_aux = use_auxiliary_latents

        self.encoder = DoubleEncoder(
            input_dim=input_dim,
            hidden_dim=hidden_dim,
            latent_dim=latent_dim,
            use_auxiliary_latents=use_auxiliary_latents,
        )

        self.decoder = AdditiveDecoder(
            latent_dim=latent_dim,
            hidden_dim=hidden_dim,
            output_dim=input_dim,
            use_auxiliary_latents=use_auxiliary_latents,
        )

        self.kernel_name = kernel_name
        self.gp_lengthscale = gp_lengthscale
        self.gp_variance = gp_variance
        self.gp_noise = gp_noise
        self.rq_alpha = rq_alpha
        self.linear_offset = linear_offset

    def reparameterize(self, mu, std):
        return mu + torch.randn_like(std) * std

    def kernel(self, x1, x2=None):
        return kernel_matrix(
            x1,
            x2,
            kernel_name=self.kernel_name,
            lengthscale=self.gp_lengthscale,
            variance=self.gp_variance,
            alpha=self.rq_alpha,
            linear_offset=self.linear_offset,
        )

    def local_interval_gp_regularizer(self, z, mu_u, std_u):
        """
        Training-time local interval GP regulariser.
        This follows the second code's marginal IntervalGP regularisation,
        extended here so each compared kernel contributes k(z_i,z_i).
        """
        K_diag = torch.diag(self.kernel(z, z)).view(-1, 1)
        K_diag = K_diag.expand_as(mu_u)

        gp_std = torch.sqrt(torch.clamp(K_diag + self.gp_noise, min=1e-8))

        lower = mu_u - std_u
        upper = mu_u + std_u

        normal = torch.distributions.Normal(
            loc=torch.zeros_like(mu_u),
            scale=gp_std,
        )

        prob = normal.cdf(upper) - normal.cdf(lower)
        prob = torch.clamp(prob, min=1e-8)

        return -torch.sum(torch.log(prob))

    def get_latent_stats(self, z, var_name="u"):
        outputs = self.encoder(z)

        if var_name == "u":
            return outputs[0], outputs[1], outputs[2]

        if var_name == "eps":
            return outputs[3], outputs[4], outputs[5]

        if var_name == "ua0":
            if not self.use_aux:
                return None, None, None
            return outputs[6], outputs[7], outputs[8]

        if var_name == "ua1":
            if not self.use_aux:
                return None, None, None
            return outputs[9], outputs[10], outputs[11]

        raise ValueError(f"Invalid var_name: {var_name}")

    def sample_latent(self, z, var_name="u"):
        mu, std, _ = self.get_latent_stats(z, var_name=var_name)

        if mu is None:
            return None

        return self.reparameterize(mu, std)

    def forward(self, z):
        if self.use_aux:
            (
                mu_u,
                std_u,
                logvar_u,
                mu_eps,
                std_eps,
                logvar_eps,
                mu_ua0,
                std_ua0,
                logvar_ua0,
                mu_ua1,
                std_ua1,
                logvar_ua1,
            ) = self.encoder(z)

            u = self.reparameterize(mu_u, std_u)
            eps = self.reparameterize(mu_eps, std_eps)
            ua0 = self.reparameterize(mu_ua0, std_ua0)
            ua1 = self.reparameterize(mu_ua1, std_ua1)

            z_recon = self.decoder(u, eps, ua0=ua0, ua1=ua1)

        else:
            mu_u, std_u, logvar_u, mu_eps, std_eps, logvar_eps = self.encoder(z)

            u = self.reparameterize(mu_u, std_u)
            eps = self.reparameterize(mu_eps, std_eps)
            ua0 = None
            ua1 = None

            z_recon = self.decoder(u, eps)

        recon_loss = F.mse_loss(z_recon, z, reduction="sum")

        # Matched to the second code
        kl_u = -0.001 * torch.sum(
            1.0 + logvar_u - mu_u.pow(2) - logvar_u.exp()
        )

        kl_eps = -0.5 * torch.sum(
            1.0 + logvar_eps - mu_eps.pow(2) - logvar_eps.exp()
        )

        gp_interval_reg = self.local_interval_gp_regularizer(
            z=z,
            mu_u=mu_u,
            std_u=std_u,
        )

        std_penalty = torch.sum(std_u ** 2)

        total_loss = (
            recon_loss
            + kl_u
            + kl_eps
            + gp_interval_reg
            + 10.0 * std_penalty
        )

        if self.use_aux:
            kl_ua0 = -0.5 * torch.sum(
                1.0 + logvar_ua0 - mu_ua0.pow(2) - logvar_ua0.exp()
            )

            kl_ua1 = -0.5 * torch.sum(
                1.0 + logvar_ua1 - mu_ua1.pow(2) - logvar_ua1.exp()
            )

            total_loss = total_loss + kl_ua0 + kl_ua1

        return total_loss, {
            "recon_loss": recon_loss.item(),
            "kl_u": kl_u.item(),
            "kl_eps": kl_eps.item(),
            "gp_interval_reg": gp_interval_reg.item(),
            "u": u,
            "eps": eps,
            "ua0": ua0,
            "ua1": ua1,
            "mu_u": mu_u,
            "std_u": std_u,
            "z_recon": z_recon,
        }


class FlexibleCausalHead(nn.Module):
    def __init__(
        self,
        latent_dim_u,
        hidden_dim=CAUSAL_HEAD_HIDDEN_DIM,
        z_y_dim=None,
        use_auxiliary_latents=False,
    ):
        super().__init__()

        self.use_aux = use_auxiliary_latents
        self.z_y_dim = z_y_dim

        input_dim = latent_dim_u + 1

        if z_y_dim is not None:
            input_dim += z_y_dim

        if use_auxiliary_latents:
            input_dim += latent_dim_u

        self.fc = nn.Linear(input_dim, hidden_dim)
        self.out = nn.Linear(hidden_dim, 1)

    def forward(self, u, t, z_y=None, ua1=None):
        t = t.view(-1, 1)

        parts = [u, t]

        if self.z_y_dim is not None:
            if z_y is None:
                raise ValueError("z_y is required but got None.")
            parts.append(z_y)

        if self.use_aux:
            if ua1 is None:
                raise ValueError("ua1 is required but got None.")
            parts.append(ua1)

        x = torch.cat(parts, dim=1)
        h = F.relu(self.fc(x))

        return self.out(h)


class CausalGPVAEwithNoise(nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_dim,
        latent_dim,
        z_y_dim=None,
        use_auxiliary_latents=False,
        kernel_name="RBF",
        gp_lengthscale=GP_LENGTHSCALE,
        gp_variance=GP_VARIANCE,
        gp_noise=GP_NOISE,
        rq_alpha=1.0,
        linear_offset=1.0,
    ):
        super().__init__()

        self.z_y_dim = z_y_dim
        self.use_aux = use_auxiliary_latents
        self.kernel_name = kernel_name
        self.rq_alpha = rq_alpha
        self.linear_offset = linear_offset

        self.vae = GPVAEwithNoise(
            input_dim=input_dim,
            hidden_dim=hidden_dim,
            latent_dim=latent_dim,
            kernel_name=kernel_name,
            gp_lengthscale=gp_lengthscale,
            gp_variance=gp_variance,
            gp_noise=gp_noise,
            use_auxiliary_latents=use_auxiliary_latents,
            rq_alpha=rq_alpha,
            linear_offset=linear_offset,
        )

        self.causal_head = FlexibleCausalHead(
            latent_dim_u=latent_dim,
            hidden_dim=CAUSAL_HEAD_HIDDEN_DIM,
            z_y_dim=z_y_dim,
            use_auxiliary_latents=use_auxiliary_latents,
        )

    def forward(self, z, t, y):
        loss_vae, vae_info = self.vae(z)

        u = vae_info["u"]
        ua1 = vae_info["ua1"] if self.use_aux else None
        z_y = make_z_y(self, z)

        y_pred = self.causal_head(
            u,
            t,
            z_y=z_y,
            ua1=ua1,
        )

        causal_loss = F.mse_loss(
            y_pred,
            y.unsqueeze(-1),
            reduction="sum",
        )

        total_loss = loss_vae + causal_loss

        return total_loss, {
            **vae_info,
            "y_pred": y_pred,
            "causal_loss": causal_loss.item(),
        }


def get_z_y_dim(chosen_version, z_dim):
    if chosen_version == "z_to_y":
        return z_dim

    if chosen_version == "split_z_to_t_and_y":
        return z_dim // 2

    return None


def get_use_aux(chosen_version):
    return chosen_version == "u_aux"


def make_z_y(model, z):
    if model.causal_head.z_y_dim == z.shape[1]:
        return z

    if model.causal_head.z_y_dim:
        return z[:, z.shape[1] // 2:]

    return None

In [ ]:
# ============================================================
# 5. Full GP posterior functions with the selected kernel

In [ ]:
# ============================================================

def full_gp_posterior_point(
    x_train,
    y_train,
    x_test,
    kernel_name="RBF",
    lengthscale=GP_LENGTHSCALE,
    variance=GP_VARIANCE,
    noise=GP_NOISE,
    alpha=1.0,
    linear_offset=1.0,
    jitter=1e-5,
):
    x_train = x_train.float()
    x_test = x_test.float()
    y_train = y_train.float().view(-1, 1)

    n_train = x_train.shape[0]
    n_test = x_test.shape[0]

    K = kernel_matrix(
        x_train,
        x_train,
        kernel_name=kernel_name,
        lengthscale=lengthscale,
        variance=variance,
        alpha=alpha,
        linear_offset=linear_offset,
    )

    K = K + noise * torch.eye(n_train, device=x_train.device)

    K_s = kernel_matrix(
        x_train,
        x_test,
        kernel_name=kernel_name,
        lengthscale=lengthscale,
        variance=variance,
        alpha=alpha,
        linear_offset=linear_offset,
    )

    K_ss = kernel_matrix(
        x_test,
        x_test,
        kernel_name=kernel_name,
        lengthscale=lengthscale,
        variance=variance,
        alpha=alpha,
        linear_offset=linear_offset,
    )

    K_ss = K_ss + jitter * torch.eye(n_test, device=x_test.device)

    current_jitter = jitter

    for _ in range(8):
        try:
            L = torch.linalg.cholesky(
                K + current_jitter * torch.eye(n_train, device=x_train.device)
            )
            break
        except RuntimeError:
            current_jitter *= 10.0
    else:
        raise RuntimeError("Cholesky failed in full_gp_posterior_point.")

    alpha_vec = torch.cholesky_solve(y_train, L)
    posterior_mean = K_s.t() @ alpha_vec

    v = torch.cholesky_solve(K_s, L)
    posterior_cov = K_ss - K_s.t() @ v
    posterior_cov = 0.5 * (posterior_cov + posterior_cov.t())

    return posterior_mean.squeeze(-1), posterior_cov


def full_gp_posterior_heteroscedastic(
    x_train,
    y_train,
    x_test,
    noise_var_train,
    kernel_name="RBF",
    lengthscale=GP_LENGTHSCALE,
    variance=GP_VARIANCE,
    min_noise=GP_NOISE,
    alpha=1.0,
    linear_offset=1.0,
    jitter=1e-5,
):
    x_train = x_train.float()
    x_test = x_test.float()
    y_train = y_train.float().view(-1, 1)

    noise_var_train = noise_var_train.float().view(-1)
    noise_var_train = torch.clamp(noise_var_train, min=min_noise)

    n_train = x_train.shape[0]
    n_test = x_test.shape[0]

    K = kernel_matrix(
        x_train,
        x_train,
        kernel_name=kernel_name,
        lengthscale=lengthscale,
        variance=variance,
        alpha=alpha,
        linear_offset=linear_offset,
    )

    K = K + torch.diag(noise_var_train)

    K_s = kernel_matrix(
        x_train,
        x_test,
        kernel_name=kernel_name,
        lengthscale=lengthscale,
        variance=variance,
        alpha=alpha,
        linear_offset=linear_offset,
    )

    K_ss = kernel_matrix(
        x_test,
        x_test,
        kernel_name=kernel_name,
        lengthscale=lengthscale,
        variance=variance,
        alpha=alpha,
        linear_offset=linear_offset,
    )

    K_ss = K_ss + jitter * torch.eye(n_test, device=x_test.device)

    current_jitter = jitter

    for _ in range(8):
        try:
            L = torch.linalg.cholesky(
                K + current_jitter * torch.eye(n_train, device=x_train.device)
            )
            break
        except RuntimeError:
            current_jitter *= 10.0
    else:
        raise RuntimeError("Cholesky failed in full_gp_posterior_heteroscedastic.")

    alpha_vec = torch.cholesky_solve(y_train, L)
    posterior_mean = K_s.t() @ alpha_vec

    v = torch.cholesky_solve(K_s, L)
    posterior_cov = K_ss - K_s.t() @ v
    posterior_cov = 0.5 * (posterior_cov + posterior_cov.t())

    return posterior_mean.squeeze(-1), posterior_cov


def latent_gp_posterior_smoothing(
    z_train,
    mu_u_train,
    z_test,
    kernel_name="RBF",
    lengthscale=GP_LENGTHSCALE,
    variance=GP_VARIANCE,
    noise=GP_NOISE,
    alpha=1.0,
    linear_offset=1.0,
):
    if mu_u_train.ndim == 1:
        mu_u_train = mu_u_train.view(-1, 1)

    latent_dim = mu_u_train.shape[1]
    means = []
    vars_diag = []

    for r in range(latent_dim):
        mean_r, cov_r = full_gp_posterior_point(
            x_train=z_train,
            y_train=mu_u_train[:, r],
            x_test=z_test,
            kernel_name=kernel_name,
            lengthscale=lengthscale,
            variance=variance,
            noise=noise,
            alpha=alpha,
            linear_offset=linear_offset,
        )

        var_r = torch.diag(cov_r).clamp_min(1e-8)

        means.append(mean_r)
        vars_diag.append(var_r)

    mean = torch.stack(means, dim=1)
    std = torch.sqrt(torch.stack(vars_diag, dim=1))

    return mean, std

In [ ]:
# ============================================================
# 6. ITE-space IntervalGP head from the second framework

In [ ]:
# ============================================================

def sample_ite_from_encoder(
    model,
    z,
    n_samples=100,
):
    model.eval()

    with torch.no_grad():
        mu_u, std_u, _ = model.vae.get_latent_stats(z, var_name="u")
        N, d_u = mu_u.shape
        device = z.device

        z_y = make_z_y(model, z)

        mu_ua1, std_ua1, _ = model.vae.get_latent_stats(z, var_name="ua1")
        has_ua1 = (mu_ua1 is not None) and (std_ua1 is not None)

        eps_u = torch.randn(n_samples, N, d_u, device=device)
        u_samples = mu_u.unsqueeze(0) + eps_u * std_u.unsqueeze(0)
        U = u_samples.reshape(-1, d_u)

        if has_ua1:
            d_ua = mu_ua1.shape[1]
            eps_ua1 = torch.randn(n_samples, N, d_ua, device=device)
            ua1_samples = mu_ua1.unsqueeze(0) + eps_ua1 * std_ua1.unsqueeze(0)
            UA1 = ua1_samples.reshape(-1, d_ua)
        else:
            UA1 = None

        if z_y is not None:
            ZY = (
                z_y.unsqueeze(0)
                .expand(n_samples, -1, -1)
                .reshape(-1, z_y.shape[1])
            )
        else:
            ZY = None

        t0 = torch.zeros(U.shape[0], device=device)
        t1 = torch.ones(U.shape[0], device=device)

        y0 = model.causal_head(U, t0, z_y=ZY, ua1=UA1).squeeze(-1)
        y1 = model.causal_head(U, t1, z_y=ZY, ua1=UA1).squeeze(-1)

        ite_samples = (y1 - y0).view(n_samples, N)

        lower_q = (1.0 - ITE_CI) / 2.0
        upper_q = 1.0 - lower_q

        ite_mean = ite_samples.mean(dim=0)
        ite_std = ite_samples.std(dim=0)
        ite_lower = ite_samples.quantile(lower_q, dim=0)
        ite_upper = ite_samples.quantile(upper_q, dim=0)

        return ite_samples, ite_mean, ite_std, ite_lower, ite_upper


def ite_space_intervalgp_head(
    model,
    z_train,
    z_test,
    n_samples=100,
    lengthscale=GP_LENGTHSCALE,
    variance=GP_VARIANCE,
    min_noise=GP_NOISE,
):
    model.eval()

    kernel_name = model.kernel_name
    alpha = model.rq_alpha
    linear_offset = model.linear_offset

    with torch.no_grad():
        _, ite_mean_train, ite_std_train, ite_lower_train, ite_upper_train = (
            sample_ite_from_encoder(
                model=model,
                z=z_train,
                n_samples=n_samples,
            )
        )

        mu_u_train, _, _ = model.vae.get_latent_stats(z_train, var_name="u")
        mu_u_test, _, _ = model.vae.get_latent_stats(z_test, var_name="u")

        (
            z_train_gp,
            mu_u_train_gp,
            ite_lower_train_gp,
            ite_upper_train_gp,
            ite_mean_train_gp,
            ite_std_train_gp,
        ) = maybe_subsample_gp_reference(
            z_train,
            mu_u_train,
            ite_lower_train.detach(),
            ite_upper_train.detach(),
            ite_mean_train.detach(),
            ite_std_train.detach(),
            max_points=MAX_GP_POINTS,
            seed=123,
        )

        u_test_smooth, u_test_smooth_std = latent_gp_posterior_smoothing(
            z_train=z_train_gp,
            mu_u_train=mu_u_train_gp,
            z_test=z_test,
            kernel_name=kernel_name,
            lengthscale=lengthscale,
            variance=variance,
            noise=min_noise,
            alpha=alpha,
            linear_offset=linear_offset,
        )

        u_train_smooth = mu_u_train_gp.detach()

        noise_var_lower = ite_std_train_gp.pow(2) + min_noise
        noise_var_upper = ite_std_train_gp.pow(2) + min_noise
        noise_var_mean = ite_std_train_gp.pow(2) + min_noise

        gp_lower_mean, gp_lower_cov = full_gp_posterior_heteroscedastic(
            x_train=u_train_smooth,
            y_train=ite_lower_train_gp,
            x_test=u_test_smooth,
            noise_var_train=noise_var_lower,
            kernel_name=kernel_name,
            lengthscale=lengthscale,
            variance=variance,
            min_noise=min_noise,
            alpha=alpha,
            linear_offset=linear_offset,
        )

        gp_upper_mean, gp_upper_cov = full_gp_posterior_heteroscedastic(
            x_train=u_train_smooth,
            y_train=ite_upper_train_gp,
            x_test=u_test_smooth,
            noise_var_train=noise_var_upper,
            kernel_name=kernel_name,
            lengthscale=lengthscale,
            variance=variance,
            min_noise=min_noise,
            alpha=alpha,
            linear_offset=linear_offset,
        )

        gp_ite_mean, gp_ite_cov = full_gp_posterior_heteroscedastic(
            x_train=u_train_smooth,
            y_train=ite_mean_train_gp,
            x_test=u_test_smooth,
            noise_var_train=noise_var_mean,
            kernel_name=kernel_name,
            lengthscale=lengthscale,
            variance=variance,
            min_noise=min_noise,
            alpha=alpha,
            linear_offset=linear_offset,
        )

        gp_lower_std = torch.sqrt(torch.diag(gp_lower_cov).clamp_min(1e-8))
        gp_upper_std = torch.sqrt(torch.diag(gp_upper_cov).clamp_min(1e-8))
        gp_ite_std = torch.sqrt(torch.diag(gp_ite_cov).clamp_min(1e-8))

        raw_lower = torch.minimum(gp_lower_mean, gp_upper_mean)
        raw_upper = torch.maximum(gp_lower_mean, gp_upper_mean)

        ite_lower_test = raw_lower - Z_VALUE_90 * gp_lower_std
        ite_upper_test = raw_upper + Z_VALUE_90 * gp_upper_std

        return {
            "ite_mean_test": gp_ite_mean,
            "ite_std_test": gp_ite_std,
            "ite_lower_test": ite_lower_test,
            "ite_upper_test": ite_upper_test,
            "u_test_smooth": u_test_smooth,
            "u_test_smooth_std": u_test_smooth_std,
            "u_test_raw": mu_u_test,
            "ite_lower_train": ite_lower_train,
            "ite_upper_train": ite_upper_train,
            "ite_mean_train": ite_mean_train,
            "ite_std_train": ite_std_train,
        }

In [ ]:
# ============================================================
# 7. Three-stage training from the second framework

In [ ]:
# ============================================================

def train_intervalgpvae(
    z_train,
    t_train,
    y_train,
    kernel_config,
    chosen_version=chosen_version,
):
    use_aux = get_use_aux(chosen_version)
    z_y_dim = get_z_y_dim(chosen_version, z_train.shape[1])

    model = CausalGPVAEwithNoise(
        input_dim=z_train.shape[1],
        hidden_dim=INTERVALGPVAE_HIDDEN_DIM,
        latent_dim=INTERVALGPVAE_LATENT_DIM,
        z_y_dim=z_y_dim,
        use_auxiliary_latents=use_aux,
        kernel_name=kernel_config["kernel_name"],
        gp_lengthscale=GP_LENGTHSCALE,
        gp_variance=GP_VARIANCE,
        gp_noise=GP_NOISE,
        rq_alpha=kernel_config.get("alpha", 1.0),
        linear_offset=kernel_config.get("linear_offset", 1.0),
    ).to(DEVICE)

    loader = DataLoader(
        TensorDataset(z_train, t_train, y_train),
        batch_size=BATCH_SIZE,
        shuffle=True,
        drop_last=False,
    )

    # --------------------------------------------------------
    # Stage 1: joint VAE + causal head training
    # --------------------------------------------------------
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR_JOINT,
        weight_decay=WEIGHT_DECAY,
    )

    model.train()

    for epoch in range(JOINT_EPOCHS):
        for z_b, t_b, y_b in loader:
            z_b = z_b.to(DEVICE)
            t_b = t_b.to(DEVICE)
            y_b = y_b.to(DEVICE)

            loss, info = model(z_b, t_b, y_b)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # --------------------------------------------------------
    # Stage 2: freeze VAE, train causal head
    # --------------------------------------------------------
    for param in model.vae.parameters():
        param.requires_grad = False

    for param in model.causal_head.parameters():
        param.requires_grad = True

    causal_optimizer = torch.optim.AdamW(
        model.causal_head.parameters(),
        lr=LR_HEAD,
        weight_decay=WEIGHT_DECAY,
    )

    for epoch in range(HEAD_EPOCHS):
        model.train()

        for z_b, t_b, y_b in loader:
            z_b = z_b.to(DEVICE)
            t_b = t_b.to(DEVICE)
            y_b = y_b.to(DEVICE)

            with torch.no_grad():
                mu_u = model.vae.get_latent_stats(z_b, var_name="u")[0]

                if model.use_aux:
                    mu_ua1, std_ua1, _ = model.vae.get_latent_stats(
                        z_b,
                        var_name="ua1",
                    )
                    ua1_b = model.vae.reparameterize(mu_ua1, std_ua1)
                else:
                    ua1_b = None

            z_y_b = make_z_y(model, z_b)

            y_pred = model.causal_head(
                mu_u.detach(),
                t_b,
                z_y=z_y_b,
                ua1=ua1_b.detach() if ua1_b is not None else None,
            )

            causal_loss = F.mse_loss(
                y_pred,
                y_b.unsqueeze(-1),
            )

            causal_optimizer.zero_grad()
            causal_loss.backward()
            causal_optimizer.step()

    # --------------------------------------------------------
    # Stage 3: freeze causal head, refine VAE
    # Important: no torch.no_grad() around causal_head here,
    # because gradients must flow through causal_head output into VAE inputs.
    # --------------------------------------------------------
    for param in model.causal_head.parameters():
        param.requires_grad = False

    for param in model.vae.parameters():
        param.requires_grad = True

    vae_optimizer = torch.optim.AdamW(
        model.vae.parameters(),
        lr=LR_VAE_REFINE,
        weight_decay=WEIGHT_DECAY,
    )

    for epoch in range(VAE_REFINE_EPOCHS):
        model.train()

        for z_b, t_b, y_b in loader:
            z_b = z_b.to(DEVICE)
            t_b = t_b.to(DEVICE)
            y_b = y_b.to(DEVICE)

            z_y_b = make_z_y(model, z_b)

            loss_vae, vae_info = model.vae(z_b)

            ua1_b = (
                model.vae.sample_latent(z_b, var_name="ua1")
                if model.use_aux
                else None
            )

            y_pred = model.causal_head(
                vae_info["u"],
                t_b,
                z_y=z_y_b,
                ua1=ua1_b,
            )

            causal_loss = F.mse_loss(
                y_pred,
                y_b.unsqueeze(-1),
                reduction="sum",
            )

            total_loss = loss_vae + causal_loss

            vae_optimizer.zero_grad()
            total_loss.backward()
            vae_optimizer.step()

    for param in model.parameters():
        param.requires_grad = True

    return model

In [ ]:
# ============================================================
# 8. Evaluation

In [ ]:
# ============================================================

def evaluate_intervalgpvae(
    model,
    z_train,
    z_test,
    true_ite_test,
    true_u_test=None,
):
    model.eval()

    intervalgp_results = ite_space_intervalgp_head(
        model=model,
        z_train=z_train,
        z_test=z_test,
        n_samples=N_ITE_SAMPLES,
        lengthscale=GP_LENGTHSCALE,
        variance=GP_VARIANCE,
        min_noise=GP_NOISE,
    )

    ite_mean = intervalgp_results["ite_mean_test"]
    ite_lower = intervalgp_results["ite_lower_test"]
    ite_upper = intervalgp_results["ite_upper_test"]

    ite_mean_np = to_numpy_1d(ite_mean)
    ite_lower_np = to_numpy_1d(ite_lower)
    ite_upper_np = to_numpy_1d(ite_upper)
    true_ite_np = to_numpy_1d(true_ite_test)

    pehe = pehe_np(ite_mean_np, true_ite_np)
    ate_error = float(abs(np.mean(ite_mean_np) - np.mean(true_ite_np)))

    interval_info = compute_interval_metrics(
        lower=ite_lower_np,
        upper=ite_upper_np,
        true_value=true_ite_np,
    )

    alpha = 1.0 - ITE_CI

    nll = approximate_nll_np(
        point=ite_mean_np,
        lower=ite_lower_np,
        upper=ite_upper_np,
        true_value=true_ite_np,
    )

    interval_score = interval_score_np(
        lower=ite_lower_np,
        upper=ite_upper_np,
        true_value=true_ite_np,
        alpha=alpha,
    )

    calibration_error = abs(interval_info["coverage"] - ITE_CI)

    metrics = {
        "PEHE": pehe,
        "ATE_error": ate_error,
        "coverage": interval_info["coverage"],
        "interval_width": interval_info["avg_length"],
        "calibration_error": calibration_error,
        "NLL": nll,
        "interval_score": interval_score,
    }

    if true_u_test is not None:
        true_u_np = to_numpy_1d(true_u_test)
        raw_u_np = to_numpy_1d(intervalgp_results["u_test_raw"])
        smooth_u_np = to_numpy_1d(intervalgp_results["u_test_smooth"])

        raw_aligned, _, _, raw_corr, raw_rmse = align_recovered_u_to_true_u(
            raw_u_np,
            true_u_np,
        )

        smooth_aligned, _, _, smooth_corr, smooth_rmse = align_recovered_u_to_true_u(
            smooth_u_np,
            true_u_np,
        )

        metrics.update(
            {
                "raw_u_corr": raw_corr,
                "raw_u_rmse": raw_rmse,
                "smooth_u_corr": smooth_corr,
                "smooth_u_rmse": smooth_rmse,
                "smooth_u_std_mean": float(
                    torch.mean(intervalgp_results["u_test_smooth_std"]).detach().cpu().item()
                ),
            }
        )

    return metrics

In [ ]:
# ============================================================
# 9. Main experiment loop

In [ ]:
# ============================================================

all_results = []
iteration_id = 0

if RUN_SYNTHETIC:
    all_settings = SYNTHETIC_SETTINGS

    if MAX_ITERATIONS is not None:
        all_settings = all_settings[:MAX_ITERATIONS]

    for setting in all_settings:
        iteration_id += 1

        setting_id = setting["setting_id"]
        proxy_funcs = setting["proxy_funcs"]
        treatment_func = setting["treatment_func"]
        outcome_func = setting["outcome_func"]

        print("\n" + "=" * 100)
        print(f"Synthetic setting {setting_id} | iteration {iteration_id}")
        print("=" * 100)

        train_data = generate_synthetic_data(
            n=N_TRAIN,
            noise_std=NOISE_STD,
            seed=GLOBAL_SEED + 100 * iteration_id,
            proxy_funcs=proxy_funcs,
            treatment_func=treatment_func,
            outcome_func=outcome_func,
        )

        test_data = generate_synthetic_data(
            n=N_TEST,
            noise_std=NOISE_STD,
            seed=GLOBAL_SEED + 100 * iteration_id + 1,
            proxy_funcs=proxy_funcs,
            treatment_func=treatment_func,
            outcome_func=outcome_func,
        )

        z_train = train_data["z"].to(DEVICE)
        t_train = train_data["t"].to(DEVICE)
        y_train = train_data["y"].to(DEVICE)

        z_test = test_data["z"].to(DEVICE)
        true_u_test = test_data["u"].to(DEVICE)
        true_ite_test = true_ite_from_outcome(
            outcome_func=outcome_func,
            u_tensor=test_data["u"],
        ).to(DEVICE)

        # Standardise Z by training statistics. This stabilises all kernels.
        z_mean = z_train.mean(dim=0, keepdim=True)
        z_std = z_train.std(dim=0, keepdim=True).clamp_min(1e-6)

        z_train = (z_train - z_mean) / z_std
        z_test = (z_test - z_mean) / z_std

        for kernel_config in KERNEL_CONFIGS:
            kernel_name = kernel_config["kernel_name"]
            display_name = kernel_config["display_name"]

            print("\n" + "-" * 100)
            print(f"Training kernel={display_name}, setting={setting_id}")
            print("-" * 100)

            start_time = time.time()

            model = train_intervalgpvae(
                z_train=z_train,
                t_train=t_train,
                y_train=y_train,
                kernel_config=kernel_config,
                chosen_version=chosen_version,
            )

            training_time = time.time() - start_time

            metrics = evaluate_intervalgpvae(
                model=model,
                z_train=z_train,
                z_test=z_test,
                true_ite_test=true_ite_test,
                true_u_test=true_u_test,
            )

            row = {
                "iteration_id": iteration_id,
                "setting_id": setting_id,
                "kernel_name": kernel_name,
                "display_name": display_name,
                "chosen_version": chosen_version,
                "latent_dim": INTERVALGPVAE_LATENT_DIM,
                "hidden_dim": INTERVALGPVAE_HIDDEN_DIM,
                "causal_head_hidden_dim": CAUSAL_HEAD_HIDDEN_DIM,
                "gp_lengthscale": GP_LENGTHSCALE,
                "gp_variance": GP_VARIANCE,
                "gp_noise": GP_NOISE,
                "batch_size": BATCH_SIZE,
                "joint_epochs": JOINT_EPOCHS,
                "head_epochs": HEAD_EPOCHS,
                "vae_refine_epochs": VAE_REFINE_EPOCHS,
                "lr_joint": LR_JOINT,
                "lr_head": LR_HEAD,
                "lr_vae_refine": LR_VAE_REFINE,
                "weight_decay": WEIGHT_DECAY,
                "ite_ci": ITE_CI,
                "max_gp_points": MAX_GP_POINTS,
                "training_time": training_time,
                **metrics,
            }

            all_results.append(row)

            print(
                f"Done: PEHE={metrics['PEHE']:.4f}, "
                f"ATE error={metrics['ATE_error']:.4f}, "
                f"coverage={metrics['coverage']:.4f}, "
                f"width={metrics['interval_width']:.4f}, "
                f"time={training_time:.2f}s"
            )

            partial_df = pd.DataFrame(all_results)
            partial_path = os.path.join(
                OUTPUT_DIR,
                "experiment_E_kernel_ablation_second_framework_partial.csv",
            )
            partial_df.to_csv(partial_path, index=False)
            print(f"Partial results saved to: {partial_path}")

# end

In [ ]:
# ============================================================

results_df = pd.DataFrame(all_results)

raw_path = os.path.join(
    OUTPUT_DIR,
    "experiment_E_kernel_ablation_second_framework_raw.csv",
)

results_df.to_csv(raw_path, index=False)

print("\n" + "=" * 100)
print(f"Raw results saved to: {raw_path}")
print("=" * 100)

summary_df = (
    results_df
    .groupby(["kernel_name", "display_name"], dropna=False)
    .agg(
        PEHE_mean=("PEHE", "mean"),
        PEHE_std=("PEHE", "std"),
        ATE_error_mean=("ATE_error", "mean"),
        ATE_error_std=("ATE_error", "std"),
        coverage_mean=("coverage", "mean"),
        coverage_std=("coverage", "std"),
        interval_width_mean=("interval_width", "mean"),
        interval_width_std=("interval_width", "std"),
        calibration_error_mean=("calibration_error", "mean"),
        calibration_error_std=("calibration_error", "std"),
        NLL_mean=("NLL", "mean"),
        NLL_std=("NLL", "std"),
        interval_score_mean=("interval_score", "mean"),
        interval_score_std=("interval_score", "std"),
        raw_u_corr_mean=("raw_u_corr", "mean"),
        smooth_u_corr_mean=("smooth_u_corr", "mean"),
        smooth_u_rmse_mean=("smooth_u_rmse", "mean"),
        training_time_mean=("training_time", "mean"),
        training_time_std=("training_time", "std"),
    )
    .reset_index()
)

kernel_order = [cfg["kernel_name"] for cfg in KERNEL_CONFIGS]
summary_df["kernel_order"] = summary_df["kernel_name"].map(
    {k: i for i, k in enumerate(kernel_order)}
)
summary_df = summary_df.sort_values("kernel_order").drop(columns=["kernel_order"])

summary_path = os.path.join(
    OUTPUT_DIR,
    "experiment_E_kernel_ablation_second_framework_summary.csv",
)

summary_df.to_csv(summary_path, index=False)

print("\nSummary:")
print(summary_df)
print(f"\nSummary saved to: {summary_path}")

In [ ]:
# ============================================================
# 11. Plotting

In [ ]:
# ============================================================

PLOT_DIR = os.path.join(OUTPUT_DIR, "figures")
os.makedirs(PLOT_DIR, exist_ok=True)

plot_metrics = [
    ("PEHE_mean", "PEHE ↓"),
    ("ATE_error_mean", "ATE error ↓"),
    ("coverage_mean", "Coverage ↑"),
    ("interval_width_mean", "Interval width ↓"),
    ("calibration_error_mean", "Calibration error ↓"),
    ("NLL_mean", "Approx. NLL ↓"),
    ("interval_score_mean", "Interval score ↓"),
    ("smooth_u_corr_mean", "Smoothed latent corr. ↑"),
    ("training_time_mean", "Training time (s) ↓"),
]

# One figure for each metric
for metric_col, ylabel in plot_metrics:
    if metric_col not in summary_df.columns:
        continue

    fig, ax = plt.subplots(figsize=(10, 5))

    x = np.arange(len(summary_df))

    ax.bar(
        x,
        summary_df[metric_col].values,
        edgecolor="black",
        linewidth=0.6,
        alpha=0.85,
    )

    ax.set_xticks(x)
    ax.set_xticklabels(summary_df["display_name"].values, rotation=30, ha="right")
    ax.set_ylabel(ylabel)
    ax.set_title(f"Kernel ablation: {ylabel}")
    ax.grid(axis="y", linestyle="--", alpha=0.5)

    if metric_col == "coverage_mean":
        ax.axhline(
            ITE_CI,
            linestyle="--",
            alpha=0.8,
            label=f"Nominal {ITE_CI:.2f}",
        )
        ax.legend()

    plt.tight_layout()

    fig_path = os.path.join(
        PLOT_DIR,
        f"experiment_E_kernel_ablation_{metric_col}.png",
    )

    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.close()

    print(f"Saved figure: {fig_path}")

# Dashboard
fig, axes = plt.subplots(3, 3, figsize=(18, 15))
axes = axes.flatten()

for ax, (metric_col, ylabel) in zip(axes, plot_metrics):
    x = np.arange(len(summary_df))

    ax.bar(
        x,
        summary_df[metric_col].values,
        edgecolor="black",
        linewidth=0.5,
        alpha=0.85,
    )

    ax.set_xticks(x)
    ax.set_xticklabels(summary_df["display_name"].values, rotation=30, ha="right", fontsize=8)
    ax.set_ylabel(ylabel)
    ax.set_title(ylabel)
    ax.grid(axis="y", linestyle="--", alpha=0.5)

    if metric_col == "coverage_mean":
        ax.axhline(
            ITE_CI,
            linestyle="--",
            alpha=0.8,
            label=f"Nominal {ITE_CI:.2f}",
        )
        ax.legend(fontsize=8)

plt.suptitle(
    "Experiment E: Kernel Ablation under the Second IntervalGP-VAE Framework",
    fontsize=18,
)

plt.tight_layout(rect=[0, 0, 1, 0.96])

dashboard_path = os.path.join(
    PLOT_DIR,
    "experiment_E_kernel_ablation_second_framework_dashboard.png",
)

plt.savefig(dashboard_path, dpi=300, bbox_inches="tight")
plt.close()

print(f"Saved dashboard figure: {dashboard_path}")

In [ ]:
# ============================================================
# 12. Compact table for paper

In [ ]:
# ============================================================

compact_summary = summary_df[
    [
        "display_name",
        "PEHE_mean",
        "ATE_error_mean",
        "coverage_mean",
        "interval_width_mean",
        "calibration_error_mean",
        "interval_score_mean",
        "smooth_u_corr_mean",
        "training_time_mean",
    ]
].copy()

compact_summary_path = os.path.join(
    OUTPUT_DIR,
    "experiment_E_kernel_ablation_second_framework_compact_summary.csv",
)

compact_summary.to_csv(compact_summary_path, index=False)

print("\nCompact summary:")
print(compact_summary)
print(f"\nCompact summary saved to: {compact_summary_path}")

print("\nAll figures saved to:")
print(PLOT_DIR)